## Importing Necessary Modules and Functions

In [58]:
import torch
import torch.nn as nn
from torch.optim import Adam

import lightning as L
from torch.utils.data import TensorDataset, DataLoader

## LSTM From Scratch

In [97]:
class LSTMbyHand(L.LightningModule):
    def __init__(self):
        super().__init__()

        L.seed_everything(seed = 42)


        # Using Normal Distribution 

        mean = torch.tensor(0.0)
        std = torch.tensor(1.0)
        # self.output_layer = nn.Linear(1,1)

        #Forget Gate
        self.w1fg = nn.Parameter(torch.normal(mean = mean, std = std), requires_grad = True)
        self.w2fg = nn.Parameter(torch.normal(mean = mean, std = std), requires_grad = True)
        self.bfg = nn.Parameter(torch.tensor(0.), requires_grad = True)

        #Input Gate
        self.w1ig1 = nn.Parameter(torch.normal(mean = mean, std = std), requires_grad = True)
        self.w2ig1 = nn.Parameter(torch.normal(mean = mean, std = std), requires_grad = True)
        self.big1 = nn.Parameter(torch.normal(mean = mean, std = std), requires_grad = True)

        self.w1ig2 = nn.Parameter(torch.normal(mean = mean, std = std), requires_grad = True)
        self.w2ig2 = nn.Parameter(torch.normal(mean = mean, std = std), requires_grad = True)
        self.big2 = nn.Parameter(torch.normal(mean = mean, std = std), requires_grad = True)

        #Output Gate
        self.w1og1 = nn.Parameter(torch.normal(mean = mean, std = std), requires_grad = True)
        self.w2og1 = nn.Parameter(torch.normal(mean = mean, std = std), requires_grad = True)
        self.bog1 = nn.Parameter(torch.normal(mean = mean, std = std), requires_grad = True)

        # uniform Distribution
                                 
        # #Forget Gate
        # self.w1fg = nn.Parameter(torch.rand(1), requires_grad = True)
        # self.w2fg = nn.Parameter(torch.rand(1), requires_grad = True)
        # self.bfg = nn.Parameter(torch.rand(1), requries_grad = True)

        # #Input Gate
        # self.w1ig1 = nn.Parameter(torch.rand(1), requires_grad = True)
        # self.w2ig1 = nn.Parameter(torch.rand(1), requires_grad = True)
        # self.big1 = nn.Parameter(torch.rand(1), requires_grad = True)

        # self.w1ig2 = nn.Parameter(torch.rand(1), requires_grad = True)
        # self.w2ig1 = nn.Parameter(torch.rand(1), requires_grad = True)
        # self.big2 = nn.Parameter(torch.rand(1), requires_grad = True)

        # #Output Gate
        # self.w1og1 = nn.Parameter(torch.rand(1), requires_grad = True)
        # self.w2og1 = nn.Parameter(torch.rand(1), requires_grad = True)
        # self.bog1 = nn.Parameter(torch.rand(1), requires_grad = True)

        # self.w1og2 = nn.Parameter(torch.rand(1), requires_grad = True)
        # self.w2og1 = nn.Parameter(torch.rand(1), requires_grad = True)
        # self.bog2 = nn.Parameter(torch.rand(1), requires_grad = True)

    def lstm_unit(self, input_value, long_memory, short_memory):

        long_remember_percent = torch.sigmoid((short_memory * self.w1fg) + input_value * self.w2fg + self.bfg)

        potential_remember_percent = torch.sigmoid((short_memory * self.w1ig1) + input_value * self.w2ig1+ self.big1)
            

        potential_memory = torch.tanh((short_memory * self.w1ig2) + input_value * self.w2ig2+ self.big2)

        updated_long_memory = long_memory * long_remember_percent + potential_remember_percent * potential_memory

        output_percent = torch.sigmoid(short_memory * self.w1og1 + input_value * self.w2og1 + self.bog1)

        updated_short_memory = torch.tanh(updated_long_memory) * output_percent

        return ([updated_long_memory, updated_short_memory])

    def forward(self, input):
        long_memory = input.new_zeros(1)
        short_memory = input.new_zeros(1)

        day1 = input[0]
        day2 = input[1]
        day3 = input[2]
        day4 = input[3]

        #Day 1
        long_memory, short_memory = self.lstm_unit(day1, long_memory, short_memory)

        #Day 2
        long_memory, short_memory = self.lstm_unit(day2, long_memory, short_memory)

        #Day 3
        long_memory, short_memory = self.lstm_unit(day3, long_memory, short_memory)

        #Day 4
        long_memory, short_memory = self.lstm_unit(day4, long_memory, short_memory)

        return short_memory
        # return self.output_layer(short_memory.unsqueeze(0)).squeeze()

    def configure_optimizers(self):
        #return Adam(self.parameters(), lr = 0.1)
        return Adam(self.parameters(), lr = 0.05)


    def training_step(self, batch):
        input_i, label_i = batch
        inputs = input_i / 10.0
        labels = label_i / 10.0
        # output_i = self.forward(input_i[0])

        # loss = ((output_i - label_i) ** 2)

        output_i = self.forward(inputs[0])
        loss = ((output_i - labels) ** 2)

        self.log("train_loss", loss)

        if (label_i == 14):
            self.log("out_0", output_i)
        else:
            self.log("out_1", output_i)

        return loss

            
        

## LSTM Abit Advanced Version From Scratch

### Generating Dataset

In [ ]:
import torch
import torch.nn as nn
import lightning as L
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

# =========================
# 🔹 Generate dataset
# =========================
def generate_data(n=500):
    inputs = []
    labels = []

    for _ in range(n):
        day1 = torch.randint(1, 10, (1,)).float()
        noise = torch.randint(0, 10, (3,)).float()

        seq = torch.cat([day1, noise])   # 4 timesteps
        label = day1 * 5                 # depends ONLY on Day 1

        inputs.append(seq)
        labels.append(label)

    return torch.stack(inputs), torch.stack(labels)

### LSTM From Scratch

In [ ]:

# =========================
# 🔹 LSTM by hand
# =========================
class LSTMbyHand(L.LightningModule):
    def __init__(self, hidden_size=4):
        super().__init__()

        L.seed_everything(42)

        self.hidden_size = hidden_size

        # Output layer
        self.output_layer = nn.Linear(hidden_size, 1)

        # Forget gate
        self.w1fg = nn.Parameter(torch.randn(hidden_size))
        self.w2fg = nn.Parameter(torch.randn(hidden_size))
        self.bfg = nn.Parameter(torch.zeros(hidden_size))

        # Input gate
        self.w1ig = nn.Parameter(torch.randn(hidden_size))
        self.w2ig = nn.Parameter(torch.randn(hidden_size))
        self.big = nn.Parameter(torch.randn(hidden_size))

        # Candidate memory
        self.w1g = nn.Parameter(torch.randn(hidden_size))
        self.w2g = nn.Parameter(torch.randn(hidden_size))
        self.bg = nn.Parameter(torch.randn(hidden_size))

        # Output gate
        self.w1og = nn.Parameter(torch.randn(hidden_size))
        self.w2og = nn.Parameter(torch.randn(hidden_size))
        self.bog = nn.Parameter(torch.randn(hidden_size))

    def lstm_unit(self, x, long_mem, short_mem):

        f = torch.sigmoid(short_mem * self.w1fg + x * self.w2fg + self.bfg)

        i = torch.sigmoid(short_mem * self.w1ig + x * self.w2ig + self.big)

        g = torch.tanh(short_mem * self.w1g + x * self.w2g + self.bg)

        long_mem = long_mem * f + i * g

        o = torch.sigmoid(short_mem * self.w1og + x * self.w2og + self.bog)

        short_mem = torch.tanh(long_mem) * o

        return long_mem, short_mem

    def forward(self, x):

        long_mem = x.new_zeros(self.hidden_size)
        short_mem = x.new_zeros(self.hidden_size)

        for t in range(4):
            long_mem, short_mem = self.lstm_unit(x[t], long_mem, short_mem)

        out = self.output_layer(short_mem)
        return out.squeeze()

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.01)

    def training_step(self, batch, batch_idx):
        x, y = batch

        # normalize
        x = x / 10.0
        y = y / 10.0

        preds = torch.stack([self.forward(seq) for seq in x])

        loss = ((preds - y.squeeze()) ** 2).mean()

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)

        return loss



### Training

In [ ]:
# =========================
# 🔹 Train
# =========================
inputs, labels = generate_data(500)

dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

model = LSTMbyHand(hidden_size=2)

trainer = L.Trainer(max_epochs=250, logger=True, enable_checkpointing=False)
trainer.fit(model, train_dataloaders=dataloader)

### Testing

In [99]:
# =========================
# 🔹 Test
# =========================
test_A = torch.tensor([7., 2., 9., 1.]) / 10.0
test_B = torch.tensor([3., 8., 5., 6.]) / 10.0

pred_A = model(test_A) * 10
pred_B = model(test_B) * 10

print("\nResults:")
print(f"Company A: Observed = 14, Predicted = {pred_A.item():.2f}")
print(f"Company B: Observed = 6, Predicted = {pred_B.item():.2f}")

## Technical Docs

What you went through is essentially a full mini–research cycle on sequence models, and it’s worth documenting it carefully because almost every issue you encountered corresponds to a fundamental concept in neural networks. I’ll rewrite your understanding as a coherent technical narrative, correcting and sharpening where needed, and filling in the deeper reasoning that was implicit in your experiments.

⸻

📘 Understanding Long-Term Dependency Learning with a Hand-Built LSTM

The goal of the project was to replicate the core idea from the StatQuest example: a model must learn a long-term dependency, where the output depends only on the first element of a sequence, while the remaining elements act as noise. In the original example from the book, the task is relatively simple: inputs and outputs are constrained to the range [0,1], and the output is binary. This makes the learning problem numerically stable and well-aligned with the activation functions used inside an LSTM.

In your adaptation, you deliberately made the problem more general and realistic by turning it into a regression task. Specifically, you defined a sequence of four values:

x = [\text{day1}, \text{noise}, \text{noise}, \text{noise}]

and the target as:

y = 2 \times \text{day1}

Later, you increased the dependency strength to:

y = 5 \times \text{day1}

This change introduced several non-trivial challenges that exposed key aspects of neural network behavior.

⸻

🔬 Initial Failure: Output Saturation

In your first implementation, the model consistently produced outputs close to 0.999, regardless of input. This behavior is not random—it is a direct consequence of the activation functions used inside the LSTM cell.

An LSTM internally uses:
	•	sigmoid activations for gates → output range [0,1]
	•	tanh activation for memory/output → range [-1,1]

Because your model returned the raw LSTM output (short-term memory), the final prediction was mathematically constrained to this limited range. However, your targets were in the range [2, 20]. This mismatch made it impossible for the model to represent the correct outputs, no matter how long it was trained. As a result, the network pushed its output toward the maximum representable value, which explains the constant value near 1.

This illustrates a fundamental principle:

Neural networks cannot learn values outside the range imposed by their final activation unless an appropriate output mapping layer is used.

⸻

⚙️ Role of Normalization

To address this, you introduced normalization by scaling both inputs and outputs (e.g., dividing by 10). This step was crucial, but its importance goes beyond simple scaling.

Without normalization, input values like 7, 8, or 9 get multiplied by weights inside the network, producing large intermediate values. When these values pass through a sigmoid function, they push it into saturation regions where outputs are extremely close to 0 or 1. In these regions, the gradient of the sigmoid becomes nearly zero, which prevents effective learning. This is a classic case of vanishing gradients due to activation saturation.

By normalizing inputs to a smaller range (e.g., [0,1]), you ensured that activations remain in their sensitive (non-saturated) regions, where gradients are meaningful and learning can occur.

Thus, normalization serves two purposes:
	1.	Keeps values within activation ranges
	2.	Preserves gradient flow during training

⸻

⚖️ Emergence of Mean Prediction Behavior

After normalization, the model began producing outputs that were approximately constant across different inputs (e.g., around 10 or later around 5). This is a well-known phenomenon in regression models trained with mean squared error (MSE).

If a model cannot identify a meaningful relationship between input and output, the optimal strategy for minimizing MSE is to predict the mean of the target values. This is not a bug—it is the mathematically optimal fallback solution.

This revealed an important insight:

When a neural network predicts the same value for all inputs, it is usually minimizing loss without learning the underlying pattern.

At this stage, your model had not yet learned that “Day 1 matters” and that other inputs are irrelevant.

⸻

🧠 Necessity of an Output Layer

Even after normalization, you observed that predictions were still inaccurate. This led you to introduce a linear output layer. This was a critical step.

The LSTM does not directly produce the final prediction. Instead, it produces a latent representation (short-term memory), which encodes information about the sequence. This representation is typically bounded (due to tanh), and therefore cannot directly represent arbitrary real-valued outputs.

The linear output layer serves as a learnable mapping:

\text{output} = W \cdot h + b

where h is the LSTM output. This allows the model to transform its internal representation into the desired output scale.

This clarifies an important conceptual point:

The LSTM learns “what to remember,” while the output layer learns “how to interpret that memory.”

⸻

🧩 Hidden Size and Model Capacity

Another key issue was model capacity. Initially, your LSTM used a hidden size of 1, meaning the memory state consisted of a single scalar. This imposed a severe limitation: the model had to store signal, filter noise, and produce output—all within a single number.

This leads to interference between responsibilities. For example, storing Day 1 accurately and ignoring noise require different transformations, which cannot be cleanly separated in a single dimension.

When you increased the hidden size to 2, the model gained the ability to represent multiple aspects of the input independently. Conceptually, this means:
	•	One dimension can specialize in storing the relevant signal (Day 1)
	•	Another dimension can absorb or filter noise

Each dimension has its own parameters and gate behaviors, allowing the model to develop parallel internal strategies. This is why increasing hidden size often leads to dramatic improvements.

A more precise definition is:

Hidden size is the dimensionality of the LSTM’s internal state, determining how much information it can store and process at each timestep.

⸻

📊 Importance of Data Quantity and Diversity

Initially, you trained on only two samples. This made it nearly impossible for the model to distinguish between signal and noise. With such limited data, the model cannot infer that Day 1 is consistently predictive while other values are irrelevant.

By generating around 500 samples with varying noise but consistent dependency on Day 1, you created a dataset where the pattern becomes statistically identifiable. The model can now observe that:
	•	Day 1 consistently correlates with the output
	•	Other values vary randomly and do not correlate

This enables the model to learn the correct dependency.

This highlights another principle:

Neural networks learn patterns through repeated exposure to variation, not from isolated examples.

⸻

🔊 Signal Strength and Learnability

When you increased the relationship from 2 \times \text{day1} to 5 \times \text{day1}, you effectively increased the signal-to-noise ratio. The influence of Day 1 on the output became stronger relative to the noise.

This made it easier for the model to detect the dependency, especially during early training when weights are not yet well-tuned.

⸻

📉 Training Dynamics and Phase Transition

Your training curves showed a long plateau followed by a sudden drop in loss, and then gradual improvement. This behavior is typical in neural networks and reflects different learning phases:
	1.	Exploration phase: The model does not yet understand the task and tries various parameter configurations.
	2.	Discovery phase: The model identifies the key dependency (Day 1 → output), causing a sharp drop in loss.
	3.	Refinement phase: The model fine-tunes parameters to improve accuracy.

This non-linear learning behavior is often described as a phase transition, where the model suddenly “discovers” the correct structure of the problem.

⸻

🧠 Final Conceptual Understanding

By the end of your experimentation, you uncovered several foundational ideas:
	•	Neural networks are constrained by activation functions and require proper scaling of data.
	•	Normalization is essential not just for scale, but for maintaining gradient flow.
	•	LSTM outputs are representations, not final predictions; a mapping layer is required.
	•	Hidden size determines representational capacity and the ability to separate signal from noise.
	•	When a model cannot learn a pattern, it defaults to predicting the mean.
	•	Data diversity is critical for learning meaningful relationships.
	•	Stronger signal makes dependencies easier to learn.
	•	Learning often occurs in sudden transitions rather than gradually.

⸻

🎯 Final Reflection

What you built is not just a working model—it is a controlled experiment demonstrating how LSTMs learn long-term dependencies. More importantly, you didn’t just follow instructions; you diagnosed failures, formed hypotheses, and iteratively corrected the system.

That process is exactly how real machine learning research and engineering works.

If you continue in this direction, the next natural step is to analyze how the memory evolves over time (i.e., inspecting the LSTM’s internal state across timesteps), which will give you direct evidence that the model is truly “remembering” Day 1 rather than approximating the output.

⸻
